# Project: Penguin Species Classification Pipeline

You'll build an end-to-end scikit-learn pipeline that cleans, encodes, imputes,
scales, and classifies penguin species from the classic `penguins.csv` dataset
(bill/flipper measurements, body mass, sex, and island).

**Dataset columns:** `species` (target), `island`, `bill_length_mm`, `bill_depth_mm`,
`flipper_length_mm`, `body_mass_g`, `sex`.

**You will practice:**
1. Loading & cleaning data
2. `ColumnTransformer` for mixed-type encoding
3. `Pipeline` / `make_pipeline` for chaining preprocessing steps
4. Train/test splitting
5. Fitting a classifier inside the pipeline
6. Evaluating with accuracy, classification report, confusion matrix
7. Cross-validation
8. (Bonus) Hyperparameter tuning with `GridSearchCV`

Work through each section in order. Sections marked **TODO** need your code.

---
### ✅ This is the SOLUTIONS notebook.

## Part 0 — Imports

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Part 1 — Load & Clean the Data

In [ ]:
url = 'https://codefinity-content-media.s3.eu-west-1.amazonaws.com/a65bbc96-309e-4df9-a790-a1eb8c815a1c/penguins.csv'

df = pd.read_csv(url)

# Remove rows with more than 1 null value
df = df[df.isna().sum(axis=1) < 2]

df.head()

In [ ]:
print(df.shape)
print(df.isna().sum())
print(df['species'].value_counts())

## Part 2 — Define Features (X) and Target (y)

In [ ]:
X, y = df.drop('species', axis=1), df['species']

## Part 3 — Build the `ColumnTransformer`

In [ ]:
ct = make_column_transformer(
    (OneHotEncoder(), ['sex', 'island']),
    remainder='passthrough'
)

## Part 4 — Build the Preprocessing Pipeline

In [ ]:
pipe = make_pipeline(ct, SimpleImputer(strategy='most_frequent'), StandardScaler())

X_transformed = pipe.fit_transform(X)

print(X_transformed.shape)
print(X_transformed[:3])

## Part 5 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## Part 6 — Full Pipeline With a Classifier

In [ ]:
full_pipe = make_pipeline(
    make_column_transformer((OneHotEncoder(handle_unknown='ignore'), ['sex', 'island']),
                             remainder='passthrough'),
    SimpleImputer(strategy='most_frequent'),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

full_pipe.fit(X_train, y_train)

## Part 7 — Evaluate on the Test Set

In [ ]:
preds = full_pipe.predict(X_test)

print('Accuracy:', accuracy_score(y_test, preds))
print()
print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))

## Part 8 — Cross-Validation

In [ ]:
scores = cross_val_score(full_pipe, X, y, cv=5, scoring='accuracy')
print(scores)
print('Mean accuracy:', scores.mean(), '+/-', scores.std())

## Part 9 (Bonus) — Hyperparameter Tuning

In [ ]:
param_grid = {
    'kneighborsclassifier__n_neighbors': [3, 5, 7, 9],
    'kneighborsclassifier__weights': ['uniform', 'distance']
}

grid = GridSearchCV(full_pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X, y)

print('Best params:', grid.best_params_)
print('Best score:', grid.best_score_)

## Part 10 — Reflection (sample answers)

1. **Why fit only on training data / CV folds?** Statistics learned during
   preprocessing (imputer fill values, scaler mean/std, one-hot categories) must
   come only from data the model is "allowed" to see during training. If we fit
   on the full dataset first, information from the test set leaks into the
   preprocessing statistics, making test performance look better than it will be
   on truly unseen data.

2. **Why does KNN benefit from `StandardScaler`?** KNN classifies a point based
   on Euclidean distance to its neighbors. A feature like `body_mass_g`
   (thousands) would dominate the distance calculation over `bill_depth_mm`
   (tens) purely due to scale, even if bill depth is more informative.
   Standardizing puts every feature on equal footing.

3. **What if `remainder='passthrough'` were omitted?** The default is
   `remainder='drop'`, so every numeric column not explicitly listed in the
   `ColumnTransformer` (bill length, bill depth, flipper length, body mass)
   would be **silently dropped**, and the model would only ever see the one-hot
   encoded `sex`/`island` columns — a major, easy-to-miss bug.